# Replication dataset - Schaefer100 cortical H dataframe

In [ ]:
import numpy as np
import os
import sys
import scipy.io as sio
import pandas as pd
from pathlib import Path

base_path = next(
    p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "CODE").is_dir() and (p / "DATA").is_dir()
)

sys.path.insert(0, str(Path(base_path) / 'CODE'))
from utils.helper_functions import remove_outliers_IQR_from_df

In [ ]:
# Parameters
num_sub = 36
num_parcels = 100
tasks = ['00', '01', '03', '05']
task_names = ["Rest", "0Back", "1Back", "2Back"]
num_tasks = len(tasks)

# Hurst
data_folder = f"{base_path}/RESULTS/replication_dataset/hurst/raw/parcelwise/FIN_D4_modwt_1_5_padding_0_decimate_0"
atlas_info_path = f"{base_path}/DATA/templates/Schaefer2018_100Parcels_7Networks_order_FSLMNI152_2mm.Centroid_RAS.csv"
raw_output_path = f"{base_path}/RESULTS/replication_dataset/hurst/dataframes/df_hurst_parcelwise_FIN_D4_modwt_1_5_padding_0_decimate_0_residuals_w_rest.csv"
cleaned_output_path = f"{base_path}/RESULTS/replication_dataset/hurst/dataframes/df_hurst_parcelwise_FIN_D4_modwt_1_5_padding_0_decimate_0_residuals_w_rest_IQR_cleaned.csv"
field = "Hursts"

os.makedirs(os.path.dirname(raw_output_path), exist_ok=True)
os.makedirs(os.path.dirname(cleaned_output_path), exist_ok=True)

## 1. Create and save dataframe

In [ ]:
# Load data
data = np.full((num_sub, num_tasks, num_parcels), np.nan)
likelihood = np.full((num_sub, num_tasks, num_parcels), np.nan)
for i in range(num_sub):
    for j in range(num_tasks):
        filename = f'{data_folder}/sub{i+1:02d}_task{tasks[j]}.mat'
        likelihood_filename = f'{data_folder}/L_sub{i+1:02d}_task{tasks[j]}.mat'
        if not os.path.exists(filename):
            print(f'path {filename} does not exist')
            continue
        mat_data = sio.loadmat(filename)
        hursts = mat_data[field]
        data[i, j, :] = hursts[0]

        if not os.path.exists(likelihood_filename):
            likelihood[i, j, :] = 1
        else:
            mat_likelihood = sio.loadmat(likelihood_filename)
            likelihoods = mat_likelihood['likelihoods']
            likelihood[i, j, :] = likelihoods[0]


# Extract network information from atlas info
parcel_info = pd.read_csv(atlas_info_path)
networks = parcel_info["ROI Name"].str.extract(r"(?:7Networks_[LR]H_)?([^_]+)")[0]

# Create DataFrame
sub_idx, task_idx, parcel_idx = np.meshgrid(
    np.arange(num_sub),
    task_names,
    np.arange(1, num_parcels + 1),
    indexing="ij"
)

df = pd.DataFrame({
    field:   data.ravel(),
    "Subject": sub_idx.ravel() + 1,   
    "Task":    task_idx.ravel(),  
    "Parcel":  parcel_idx.ravel(),
    "Network": np.tile(networks.values, num_sub * num_tasks),
    "Likelihood": likelihood.ravel()
})

df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)

# Save DataFrame
df.to_csv(raw_output_path, index=False)

## 2. Remove outliers

In [ ]:
# Load the DataFrame and prepare for analysis
hurst_df = pd.read_csv(raw_output_path)

hurst_df["Task"] = pd.Categorical(
    hurst_df["Task"],
    categories=task_names,
    ordered=True
)

parcels = np.unique(hurst_df['Parcel'])
networks = np.unique(hurst_df['Network'])
tasks = np.unique(hurst_df['Task'])

# Clean the DataFrame by removing outliers and save the cleaned version
cleaned_hurst_df = remove_outliers_IQR_from_df(hurst_df, threshold=1.5, value_col=field)
cleaned_hurst_df.to_csv(cleaned_output_path, index=False)